# DynaPrompt - DiT Image Generation (Google Colab)

**Dynamic Prompt Guidance for Diffusion Transformers**

**Team:** Charles Hou (ch3889), Max Kim (zk2295), Swapnil Banerjee (sb5041)  
**Course:** EECS 6694 Deep Learning

---

## Setup Instructions

1. **Enable GPU:** Runtime → Change runtime type → GPU (T4)
2. **Run all cells sequentially**
3. **First run downloads ~2GB DiT model**

---

## Step 1: Verify GPU

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

## Step 2: Clone Repository

In [ ]:
import os

# Clone repository
if not os.path.exists('/content/6694-DynaPrompt'):
    print("📥 Cloning repository...")
    !git clone https://github.com/ch3889/6694-DynaPrompt.git
    print("✓ Cloned")
else:
    print("✓ Already exists")

# Navigate and checkout swap-colab branch
%cd /content/6694-DynaPrompt
!git checkout swap-colab
!git pull origin swap-colab

%cd 6694-DynaPrompt

# Verify
print(f"\n📂 Directory: {os.getcwd()}")
print("📌 Branch:", end=" ")
!git branch --show-current
print("\n📁 Scripts:")
!ls scripts/ | grep dit

## Step 3: Install Dependencies

In [ ]:
!pip install -q diffusers transformers accelerate
!pip install -q git+https://github.com/openai/CLIP.git
print("✓ Dependencies installed")

## Step 4: Generate Images with DiT (Real Model)

In [ ]:
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
import torch

print("Loading DiT-XL/2-256 model...")
print("First run downloads ~2GB model weights\n")

pipe = DiTPipeline.from_pretrained(
    "facebook/DiT-XL-2-256",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

print(f"✓ DiT model loaded on {device}")

## Step 5: Generate Single Image

**Note:** DiT is class-conditional (ImageNet classes), not text-to-image.  
Class examples: 207=Golden Retriever, 281=Tabby Cat, 207=Dog

In [ ]:
# ImageNet class labels (0-999)
class_label = 207  # Golden Retriever
# Other options: 281 (Tabby Cat), 388 (Giant Panda), 417 (Balloon)

print(f"Generating image for class {class_label}...\n")

image = pipe(
    class_labels=[class_label],
    num_inference_steps=25,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

# Display
from IPython.display import display
display(image)

# Save
!mkdir -p outputs
image.save(f"outputs/dit_class_{class_label}.png")
print(f"\n✓ Saved to outputs/dit_class_{class_label}.png")

## Step 6: Generate Multiple Classes

In [ ]:
# ImageNet classes
classes = {
    207: "Golden Retriever",
    281: "Tabby Cat",
    388: "Giant Panda",
    417: "Balloon",
    933: "Cheeseburger"
}

print(f"Generating {len(classes)} images...\n")

for class_id, class_name in classes.items():
    print(f"Class {class_id}: {class_name}")
    
    image = pipe(
        class_labels=[class_id],
        num_inference_steps=25,
        generator=torch.Generator(device=device).manual_seed(class_id)
    ).images[0]
    
    display(image)
    image.save(f"outputs/dit_{class_id}_{class_name.replace(' ', '_')}.png")
    print()

## Step 7: Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/DiT_Outputs
!cp -r outputs/* /content/drive/MyDrive/DiT_Outputs/

print("✓ Outputs saved to Google Drive: MyDrive/DiT_Outputs/")

---

## ImageNet Class Reference

Common classes for testing:
- **207:** Golden Retriever
- **281:** Tabby Cat  
- **388:** Giant Panda
- **417:** Balloon
- **933:** Cheeseburger
- **980:** Volcano

Full list: https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a

---

## Text-to-Image with DiT: PixArt-α

**PixArt-α is a DiT-based text-to-image model** (unlike DiT-XL which is class-conditional)

In [ ]:
# Load PixArt-α (DiT-based text-to-image)
from diffusers import PixArtAlphaPipeline
import torch

print("Loading PixArt-α (DiT-based text-to-image)...")
print("Downloading ~1GB model weights\n")

pixart_pipe = PixArtAlphaPipeline.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
pixart_pipe = pixart_pipe.to(device)

print(f"✓ PixArt-α loaded on {device}")

In [ ]:
# Generate from TEXT PROMPT with DiT
from IPython.display import display

prompt = "A blue cat sitting on a red chair with a yellow ball"

print(f"Generating: '{prompt}'...")

image = pixart_pipe(
    prompt=prompt,
    num_inference_steps=20,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

display(image)
image.save("outputs/pixart_text_to_image.png")
print("\n✓ PixArt DiT text-to-image complete!")

### Generate Multiple Text Prompts with PixArt (DiT)

---

## DynaPrompt: Auto-Reweighting with Attention Boosting

**Adaptive per-token attention amplification** - automatically boosts under-attended words

In [ ]:
import matplotlib.pyplot as plt

test_prompt = "A purple elephant wearing a pink hat in a green forest"
seed = 12345

# Extract critical words automatically
critical_words_auto = extract_critical_words(test_prompt)
print(f"Test prompt: {test_prompt}")
print(f"Auto-detected critical words: {critical_words_auto}\n")

# 1. Vanilla generation (no boosting)
print("1️⃣ Vanilla PixArt-α (no DynaPrompt)")
booster.disable()
img_vanilla = pixart_pipe(
    prompt=test_prompt,
    num_inference_steps=20,
    generator=torch.Generator(device=device).manual_seed(seed)
).images[0]

# 2. DynaPrompt with auto-reweighting
print("\n2️⃣ DynaPrompt with auto-reweighting")
token_indices = get_critical_token_indices(pixart_pipe.tokenizer, test_prompt, critical_words_auto)
print(f"Boosting token indices: {token_indices}")

booster.set_indices(token_indices)
booster.enable()

img_dynaprompt = pixart_pipe(
    prompt=test_prompt,
    num_inference_steps=20,
    generator=torch.Generator(device=device).manual_seed(seed)
).images[0]

booster.disable()

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(img_vanilla)
axes[0].set_title("Vanilla PixArt-α", fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(img_dynaprompt)
axes[1].set_title("DynaPrompt Auto-Reweighting", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig("outputs/comparison_vanilla_vs_dynaprompt.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Comparison saved!")

### Compare: Vanilla vs DynaPrompt

In [ ]:
# Example: Complex compositional prompt
prompt = "A blue cat sitting on a red chair with a yellow ball"
critical_words = ["blue", "cat", "red", "chair", "yellow", "ball"]

print(f"Prompt: {prompt}")
print(f"Critical words: {critical_words}")

# Get token indices
token_indices = get_critical_token_indices(pixart_pipe.tokenizer, prompt, critical_words)
print(f"Token indices to boost: {token_indices}\n")

# Configure booster
booster.set_indices(token_indices)
booster.enable()

print("Generating with DynaPrompt auto-reweighting...")
image_boosted = pixart_pipe(
    prompt=prompt,
    num_inference_steps=20,
    generator=torch.Generator(device=device).manual_seed(42)
).images[0]

booster.disable()

display(image_boosted)
image_boosted.save("outputs/pixart_dynaprompt_boosted.png")
print("\n✓ DynaPrompt generation complete!")

### Generate with DynaPrompt Auto-Reweighting

In [ ]:
import torch
from typing import List

class AttentionBooster:
    """
    DynaPrompt's adaptive attention booster.
    Applies automatic per-token scaling based on current attention levels.
    """
    def __init__(self, boost_factor: float = 6.0, max_multiplier: float = 18.0):
        self.boost_factor = boost_factor
        self.max_multiplier = max_multiplier
        self.indices: List[int] = []
        self.enabled: bool = False

    def set_indices(self, indices: List[int]):
        self.indices = list(indices)

    def enable(self):
        self.enabled = True

    def disable(self):
        self.enabled = False

    def apply(self, attn: torch.Tensor) -> torch.Tensor:
        """
        Apply adaptive boosting to attention tensor.
        Auto-reweighting: boost varies by current attention level
        """
        if not self.enabled or len(self.indices) == 0:
            return attn

        boosted = attn.clone()

        # Adaptive boost based on current mean attention per token
        for idx in self.indices:
            if idx >= boosted.shape[-1]:
                continue
            
            current = boosted[:, :, idx].mean().item()
            
            # Auto-reweighting logic: weaker tokens get stronger boost
            if current < 1e-3:
                mult = self.boost_factor * 3.0  # Very weak: 3x boost
            elif current < 5e-3:
                mult = self.boost_factor * 2.0  # Weak: 2x boost
            elif current < 1e-2:
                mult = self.boost_factor * 1.5  # Moderate: 1.5x boost
            else:
                mult = self.boost_factor          # Normal: standard boost
            
            mult = min(mult, self.max_multiplier)
            boosted[:, :, idx] *= mult

        # Renormalize to maintain attention probability distribution
        boosted = boosted / boosted.sum(dim=-1, keepdim=True)
        return boosted

print("✓ AttentionBooster class defined")

In [ ]:
# Generate multiple prompts WITH DynaPrompt auto-reweighting
prompts = [
    "A blue cat sitting on a red chair with a yellow ball",
    "A golden retriever playing with a red ball in a snowy park",
    "A steampunk robot drinking coffee in a cafe",
    "An astronaut riding a horse in space"
]

for i, prompt in enumerate(prompts, 1):
    print(f"\n[{i}/{len(prompts)}] {prompt}")
    
    # Auto-detect critical words
    critical_words = extract_critical_words(prompt)
    token_indices = get_critical_token_indices(pixart_pipe.tokenizer, prompt, critical_words)
    
    # Enable boosting for this prompt
    booster.set_indices(token_indices)
    booster.enable()
    
    img = pixart_pipe(
        prompt=prompt,
        num_inference_steps=20,
        generator=torch.Generator(device=device).manual_seed(i * 42)
    ).images[0]
    
    booster.disable()
    
    display(img)
    img.save(f"outputs/pixart_dynaprompt_{i:02d}.png")
    print(f"✓ Generated with DynaPrompt boosting on: {critical_words}")